#Feature Customers

##Data Definitions

In [0]:
CATALOG_NAME = "ml_training_dev"
SOURCE_SCHEMA_NAME = "gold"
TARGET_SCHEMA_NAME = "feature"
SOURCE_TABLE_NAME = "customers_orders_category"
TARGET_TABLE_NAME = "customers"

##Create Catalog and schema

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS {}.{}".format(CATALOG_NAME, TARGET_SCHEMA_NAME))

##Import Libraries

In [0]:
from pyspark.sql.functions import current_date, count,sum, avg, round, max, to_date,col, min, countDistinct, row_number
from pyspark.sql.window import Window

##Reading Source Tables

In [0]:
df_feature_table = spark.table(f"{CATALOG_NAME}.{SOURCE_SCHEMA_NAME}.{SOURCE_TABLE_NAME}").drop("loadDate")

##Transformations

In [0]:
customer_features = (df_feature_table
            .dropDuplicates(["customer_unique_id", "order_id", "order_item_id"])
            .groupBy("customer_unique_id",
                    "customer_state")
            .agg(countDistinct("order_id").alias("total_orders"),
                 count("order_item_id").alias("total_items"),
                 round(sum("price"),2).alias("total_spent"),
                 round(avg("price"), 2).alias("avg_item_price"),
                 round(avg("freight_value"), 2).alias("avg_freight"),
                 countDistinct("product_id").alias("unique_products"),
                 countDistinct("product_category_name").alias("unique_categories")
                 
                 )
).withColumn(
        "avg_order_value",
        round(col("total_spent") / col("total_orders"),2)
    )

In [0]:
category_count = (
    df_feature_table
    .groupBy(
        "customer_unique_id",
        "product_category_name"
    )
    .agg(
        count("*").alias("category_purchases")
    )
)

window_category = Window.partitionBy(
    "customer_unique_id"
).orderBy(
    col("category_purchases").desc(),
    col("product_category_name").asc()
)

favorite_category = (
    category_count
    .withColumn(
        "rank",
        row_number().over(window_category)
    )
    .filter(col("rank") == 1)
    .select(
        "customer_unique_id",
        col("product_category_name").alias("favorite_category")
    )
)

In [0]:
# favorite_category.display()

In [0]:
df_final = customer_features.join(
        favorite_category,
        on="customer_unique_id",
        how="left"
    )

In [0]:
# df_final.display()

##Writting table

In [0]:
df_final.write.mode("overwrite").saveAsTable(f"{CATALOG_NAME}.{TARGET_SCHEMA_NAME}.{TARGET_TABLE_NAME}")
print("table successfully created!")

In [0]:
# Cliente A ── Produto 1
#          ├─ Produto 2
#          └─ Produto 3

# Cliente B ── Produto 1
#          ├─ Produto 2
#          └─ Produto 4

# A → recomenda Produto 4

In [0]:
# Cliente gosta de:

# categoria = beleza
# preço = 50-100
# frete = baixo

In [0]:
# customer_id
# product_id
# customer_features
# product_features
# interaction_features
#         │
#         ▼
# Logistic Regression
#         │
#         ▼
# P(buy product | customer)



# Cliente A + Produto X → 0.82
# Cliente A + Produto Y → 0.63
# Cliente A + Produto Z → 0.12



# X
# Y